# 05 - Mechanistic extras
**Paper:** Section 5 (adversarial suffixes) + probes. **Exercise used:** 6.
1. Where in depth does the direction separate harmful from harmless?
2. What does the direction read as through the unembedding?
3. Which attention heads write it?
4. Does an adversarial suffix suppress it, and do those heads look at the suffix instead of the instruction?

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/YOUR_GITHUB_USERNAME/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
try:  # autoreload picks up edits to src/ without restarting
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:  # Colab's IPython autoreload imports `imp`, which Python 3.12+ removed
    pass
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

In [ ]:
from refusal_direction import artifacts
from refusal_direction.mechanistic import projection_by_layer, head_contributions, attention_from_position, logit_lens_direction
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)
H, S = tok(data["harmful_val"]), tok(data["harmless_val"])

## 1. Projection onto r across layers (last token)

In [ ]:
ph = projection_by_layer(model, *H, direction).cpu()
ps = projection_by_layer(model, *S, direction).cpu()
plt.figure(figsize=(8, 3))
for p, lab in [(ph, "harmful"), (ps, "harmless")]:
    plt.plot(p.mean(0), label=lab); plt.fill_between(range(p.shape[1]), p.mean(0) - p.std(0), p.mean(0) + p.std(0), alpha=.2)
plt.axvline(meta["layer"], c="gray", ls=":"); plt.xlabel("layer"); plt.ylabel("x . r_hat"); plt.legend(); plt.show()

## 2. Logit lens on r

In [ ]:
ll = logit_lens_direction(model, direction, k=15)
pd.DataFrame({"top": [t for t, _ in ll["top"]], "bottom": [t for t, _ in ll["bottom"]]})

## 3. Which heads write the direction?

In [ ]:
hc = head_contributions(model, *H, direction).mean(0).cpu()   # [n_layers, n_heads]
plt.figure(figsize=(6, 6)); plt.imshow(hc, cmap="RdBu", vmin=-hc.abs().max(), vmax=hc.abs().max())
plt.colorbar(); plt.xlabel("head"); plt.ylabel("layer"); plt.title("mean head output . r_hat (harmful)"); plt.show()
flat = hc.flatten().topk(8)
top_heads = [(int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads) for i in flat.indices]
top_heads

## 4. Adversarial suffix (paper Section 5)
You need a suffix that jailbreaks *your* model. Option A (recommended): optimize one with GCG via `nanogcg` (~10-20 min on a T4). Option B: skip to the random-suffix control to get the plumbing working first.

The paper optimizes one universal suffix over many prompts; nanoGCG optimizes per prompt, which is fine for a first look.

In [ ]:
# Option A: GCG. Uses a separate HF copy of the model (nanogcg doesn't speak TransformerLens).
# --no-deps: nanogcg pins transformers<=4.47.1, which would break transformer_lens 2.18 mid-session
%pip install -q --no-deps nanogcg
import nanogcg
from transformers import AutoModelForCausalLM, AutoTokenizer
hf_tok = AutoTokenizer.from_pretrained(cfg.model_name)
# Match OUR prompt format exactly (no auto-injected Qwen system prompt):
hf_tok.chat_template = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
                        "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
hf_model = AutoModelForCausalLM.from_pretrained(cfg.model_name, torch_dtype=model.cfg.dtype).to(model.cfg.device)
target_instr = data["harmful_val"][0]
gcg_cfg = nanogcg.GCGConfig(num_steps=250, search_width=256, topk=256, seed=cfg.seed, verbosity="WARNING")
res = nanogcg.run(hf_model, hf_tok, target_instr + " {optim_str}", "Sure, here", gcg_cfg)
SUFFIX = " " + res.best_string
del hf_model; torch.cuda.empty_cache()
print(repr(SUFFIX))

In [ ]:
# Option B / control: random tokens of similar length
g = torch.Generator().manual_seed(0)
RANDOM_SUFFIX = " " + model.tokenizer.decode(torch.randint(1000, 30000, (20,), generator=g))
SUFFIX = globals().get("SUFFIX", RANDOM_SUFFIX)

In [ ]:
instrs = [target_instr] if "target_instr" in globals() else data["harmful_val"][:8]
conds = {"harmful": "", "harmful+adv_suffix": SUFFIX, "harmful+random_suffix": RANDOM_SUFFIX}
for name, sfx in conds.items():
    T = tok(instrs, sfx)
    out = generate(model, *T, max_new_tokens=40, show_progress=False)[0]
    proj = projection_by_layer(model, *T, direction)[:, meta["layer"]].mean().item()
    print(f"{name:24s} proj@layer{meta['layer']}={proj:7.2f} | {out[:100]!r}")

In [ ]:
# Where do the top heads look? Attention mass from the last token on the instruction vs. the suffix.
n_eoi, n_sfx = len(eoi_positions(model)), len(model.tokenizer.encode(SUFFIX, add_special_tokens=False))
T = tok(instrs, SUFFIX)
pat = attention_from_position(model, *T, top_heads)  # [batch, heads, key_pos]
seq = T[0].shape[1]
sfx_slice = slice(seq - n_eoi - n_sfx, seq - n_eoi)   # approx: token boundaries can shift by one
instr_slice = slice(0, seq - n_eoi - n_sfx)
pd.DataFrame({"head": [f"L{l}H{h}" for l, h in top_heads],
              "attn_on_instruction": pat[..., instr_slice].sum(-1).mean(0).cpu(),
              "attn_on_suffix": pat[..., sfx_slice].sum(-1).mean(0).cpu()})
# Compare with the same table computed with RANDOM_SUFFIX.

## Questions to write up
1. At which layer does harmful/harmless separation appear? Is it before the selected layer?
2. Does the logit lens show refusal-flavoured tokens, or is r mostly not a direct "say I'm sorry" direction? What does that suggest about how refusal is implemented downstream?
3. With the adversarial suffix: does the projection drop, and do the top heads shift attention onto the suffix (paper Fig. 5)? Does the random suffix do the same? If not, what does the suffix do that random tokens don't?